## RAG step by step


**RAG 구조**
```
school.txt
   ↓
Chunk + Overlap                 # Level 3
   ↓
Embedding
   ↓
Chroma Vector DB                # Level 2
   │
   ├─ Metadata Filter           # Level 4
   │
   ├─ Vector Search ─┐
   │                 ├─ RRF     # Level 5
   └─ BM25 Search ───┘
             ↓
        Reranker                # Level 6
             ↓
       Top-K Evidence
             ↓
        LLM Answer
        + [C1][C2]              # Level 7
             ↓
     Grounding Check            # Level 8
             ↓
     FAIL? Query Rewrite
             ↓
         재검색                  # Level 9
```


## 기본 설정

In [2]:
import os
import re
import numpy as np
import chromadb

from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

C:\Users\nhkim\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

DATA_PATH = "data/school2.txt"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# multilingual reranker
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

OPENAI_MODEL = os.getenv("OPENAI_MODEL")

In [4]:
# Embedding Model
embedder = SentenceTransformer(EMBEDDING_MODEL)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1982.50it/s]


## Level 3 : Chunk + Overlap

문서를 검색하기 좋게 나눈 하나의 텍스트 조각

### 1) section 별로 데이터 분류하는 함수

In [5]:
def load_sections(path):
    """
    [도서관], [학사] 같은 section을 읽는다.
    """

    sections = []

    current_section = "일반"
    buffer = []

    with open(path, "r", encoding="utf-8") as f:

        for raw_line in f:

            line = raw_line.strip()

            if not line:
                continue

            # [도서관]
            match = re.fullmatch(r"\[(.+)]", line)

            if match:

                # 이전 section 저장
                if buffer:
                    sections.append(
                        (
                            current_section,
                            " ".join(buffer)
                        )
                    )

                current_section = match.group(1)
                buffer = []

            else:
                buffer.append(line)

    # 마지막 section
    if buffer:
        sections.append(
            (
                current_section,
                " ".join(buffer)
            )
        )

    return sections

결과물 예시 :

[
    ("도서관", "문장 A 문장 B"),
    ("학사", "문장 C 문장 D")
]

### 2) 내용 일부 겹치게 자르는 함수

In [6]:
def chunk_with_overlap(
    text,
    chunk_size=35,
    overlap=10
):
    """
    단어 기준 chunking.

    예:
    chunk_size = 35
    overlap = 10

    chunk1:
    [0 ~ 34]

    chunk2:
    [25 ~ 59]

    앞 chunk의 마지막 10단어가
    다음 chunk에 다시 들어간다.
    """

    if overlap >= chunk_size:
        raise ValueError(
            "overlap은 chunk_size보다 작아야 합니다."
        )

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = min(
            start + chunk_size,
            len(words)
        )

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        if end == len(words):
            break

        # overlap
        start = end - overlap

    return chunks

결과물 예시 :

[
    "나는 오늘 학교 도서관에 갔다",
    "도서관에 갔다 도서관에는 많은 학생들이",
    "많은 학생들이 공부하고 있었다 나는",
    "있었다 나는 인공지능 관련 책을",
    "관련 책을 찾아서 두 시간",
    "두 시간 동안 읽었다"
]

### 3) 종합 chunking
앞에서 만든 두 함수를 연결해서 Chroma에 넣기 좋은 문서 형태로 만드는 단계

In [7]:
def build_documents(path):
    # 원본 txt 파일을 읽고, section별로 나누기
    sections = load_sections(path)

    documents = []

    global_index = 0 # chunk마다 고유 ID 부여 용 (chunk-000, chunk-001, ...)

    for section, text in sections:
        # 각 section을 chunk로 나누기
        chunks = chunk_with_overlap(
            text,
            chunk_size=35,
            overlap=10
        )

        for local_index, chunk in enumerate(chunks):

            documents.append(
                {
                    "id": f"chunk-{global_index:03d}",

                    "text": chunk,

                    "metadata": {
                        "source": os.path.basename(path),
                        "section": section,
                        "chunk_index": local_index
                    }
                }
            )

            global_index += 1

    return documents


documents = build_documents(DATA_PATH)

documents 결과 예시 : 
```
documents = [
    {
        "id": "chunk-000",
        "text": "도서관 첫 번째 내용...",
        "metadata": {
            "source": "school.txt",
            "section": "도서관",
            "chunk_index": 0
        }
    },

    {
        "id": "chunk-001",
        "text": "도서관 두 번째 내용...",
        "metadata": {
            "source": "school.txt",
            "section": "도서관",
            "chunk_index": 1
        }
    },
    ...
]
```
메타데이터 중 section을 나중에 Chroma에서:

where={"section": "도서관"} 처럼 metadata filtering할 때 사용할 수 있음

In [8]:
print("\n===== CHUNKS =====")

for doc in documents:
    print(
        doc["id"],
        doc["metadata"],
        doc["text"]
    )



===== CHUNKS =====
chunk-000 {'source': 'school2.txt', 'section': '도서관', 'chunk_index': 0} 학교 도서관은 오전 9시부터 오후 9시까지 운영한다. 도서 대출 기간은 최대 14일이다. 학생은 한 번에 최대 5권까지 대출할 수 있다. 대출한 도서는 반납 예정일까지 반납해야 한다. 연체한 학생은 연체 기간 동안 추가 대출이 제한될 수
chunk-001 {'source': 'school2.txt', 'section': '도서관', 'chunk_index': 1} 한다. 연체한 학생은 연체 기간 동안 추가 대출이 제한될 수 있다.
chunk-002 {'source': 'school2.txt', 'section': '학사', 'chunk_index': 0} 졸업 요건은 총 130학점 이상 이수하는 것이다. 전공 필수 과목을 모두 이수해야 졸업할 수 있다. 졸업 예정자는 졸업 심사를 받아야 한다.
chunk-003 {'source': 'school2.txt', 'section': '학생식당', 'chunk_index': 0} 학생식당은 오전 11시부터 오후 7시까지 운영한다. 학생증으로 식권을 구매할 수 있다.


## 2. Embedding

문장을 숫자 벡터로 바꾸기

In [9]:
texts = [
    doc["text"]
    for doc in documents
]

document_embeddings = embedder.encode(
    texts,
    normalize_embeddings=True # 각 벡터의 길이를 1로 정규화
)

In [10]:
print(document_embeddings.shape)
print(document_embeddings)

(4, 384)
[[ 0.02372567  0.00852014 -0.06223107 ... -0.00274532 -0.00743411
  -0.06587334]
 [ 0.03385396  0.05757532 -0.03343306 ... -0.00312169  0.05480455
  -0.03820291]
 [ 0.15395117  0.05976061  0.02291638 ...  0.06491773  0.01491374
   0.0440379 ]
 [ 0.07117082  0.01845513 -0.02492453 ... -0.00898683 -0.09403856
  -0.01132436]]


## Level 1 : Cosine Similarity

In [11]:
def cosine_search(
    question,
    top_k=3
):

    query_embedding = embedder.encode(
        question,
        normalize_embeddings=True
    )

    # normalize 되어 있기 때문에
    # dot product == cosine similarity

    scores = (
        document_embeddings
        @ query_embedding    # 행렬 곱
    )

    # argsort() : 점수를 작은 순서대로 정렬했을 때의 index를 반환.
    # [::-1] : 내림차순 정렬 (반대로 정렬하기)
    # [:top_k] : 상위 top_k개만 가져오기
    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in indices:

        # 해당 인덱스의 document를 가져오기
        doc = documents[index]  

        results.append(
            {
                **doc,
                "score": float(scores[index])
            }
        )

    return results

결과 예시 : 
```
{
    "id": "chunk-000",
    "text": "도서관은 오후 9시까지 운영한다.",
    "metadata": {
        "section": "도서관"
    },
    "score": 0.91
}
```

## Level 2 : Chroma Vector DB
## Level 4 : Metadata filtering

Chroma에는 txt 파일에서 만든 chunk들의 임베딩이 저장되어 있고,
사용자 질문의 임베딩은 검색용으로 Chroma에 전달한다.
그러면 Chroma가 저장된 문서 임베딩들과 비교해서 유사한 chunk와 distance 등을 반환한다.

이게 바로 Vector DB를 사용하는 이유다. 직접 NumPy로 모든 벡터를 비교하지 않고, Chroma가 저장·검색·필터링을 맡는다.

In [12]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="school_rag"
)


collection.upsert(
    ids=[
        doc["id"]
        for doc in documents
    ],

    documents=[
        doc["text"]
        for doc in documents
    ],

    embeddings=document_embeddings.tolist(),

    metadatas=[
        doc["metadata"]
        for doc in documents
    ]
)  # [1]


def vector_search(
    question,
    top_k=10,
    where=None
):

    query_embedding = embedder.encode(
        question,
        normalize_embeddings=True
    )

    query_args = {  # [2]
        "query_embeddings": [
            query_embedding.tolist()
        ],
        "n_results": min(
            top_k,
            len(documents)
        ),
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }
    # =====================================================
    # Level 4
    # Metadata filtering
    # =====================================================

    if where is not None:
        query_args["where"] = where  # [3]

    # Chroma에 실제 검색 요청
    result = collection.query(
        **query_args
    )  # [4]

    hits = []

    if not result["ids"]:
        return hits

    for (
        doc_id,
        text,
        metadata,
        distance
    ) in zip(  # zip()은 여러 리스트의 같은 위치에 있는 값들을 하나씩 묶어주는 함수
        result["ids"][0],
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0]
    ):

        hits.append(
            {
                "id": doc_id,
                "text": text,
                "metadata": metadata,
                "vector_distance": float(distance)
            }
        )

    return hits # [5]


[1] Chroma 저장 예시
```
collection: school_rag

chunk-000
├─ id
│  └─ "chunk-000"
│
├─ document
│  └─ "학교 도서관은 오전 9시부터 오후 9시까지 운영한다."
│
├─ embedding
│  └─ [0.12, -0.31, 0.44, 0.08, ...]
│
└─ metadata
   ├─ source: "school.txt"
   ├─ section: "도서관"
   └─ chunk_index: 0
```

[2] query_args 결과 예시 : 
```
{
    "query_embeddings": [
        [0.12, -0.31, 0.44, ...]
    ],

    "n_results": 3,

    "include": [
        "documents",
        "metadatas",
        "distances"
    ]
}
```

[3] Level 4의 meta data filtering 적용 시 query_args 예시 : 

```
{
    "query_embeddings": [[...]],
    "n_results": 3,
    "include": [
        "documents",
        "metadatas",
        "distances"
    ],
    "where": {
        "section": "도서관"
    }
}
```

[4] result 결과 예시 : 
```
result = {
    "ids": [
        [
            "chunk-000",
            "chunk-001"
        ]
    ],

    "documents": [
        [
            "도서관은 오전 9시부터 오후 9시까지 운영한다.",
            "도서 대출 기간은 최대 14일이다."
        ]
    ],

    "metadatas": [
        [
            {
                "section": "도서관",
                "chunk_index": 0
            },
            {
                "section": "도서관",
                "chunk_index": 1
            }
        ]
    ],

    "distances": [
        [
            0.08,
            0.27
        ]
    ]
}
```

[5] hits 결과 예시 :
```
[
    {
        "id": "chunk-000",
        "text": "도서관은 오후 9시까지 운영한다.",
        "metadata": {
            "section": "도서관"
        },
        "vector_distance": 0.08
    },

    {
        "id": "chunk-001",
        "text": "대출 기간은 14일이다.",
        "metadata": {
            "section": "도서관"
        },
        "vector_distance": 0.27
    }
]
```

## Level 5 : BM25

앞에서 Chroma는 문장을 임베딩해서 의미가 비슷한 문서를 찾음. 이번에는 반대로 문장 안에 어떤 단어가 들어있는지를 기준으로 찾음.

In [13]:
# 단어 단위로 쪼개는 tokenizer
def tokenize(text):
    """
    아주 단순한 tokenizer.

    실제 한국어 서비스라면
    Kiwi / MeCab 등을 붙이는 것이 좋다.
    (단순 tokenizer는 "도서관" != "도서관은" 으로 보기 때문에 정확도 낮음
    => Kiwi나 MeCab 같은 형태소 분석기로 ["도서관", "은"] 처럼 자름)
    """

    return re.findall(
        r"[가-힣A-Za-z0-9]+",
        text.lower()
    )

# 모든 문서 chunk를 tokenize
tokenized_corpus = [
    tokenize(doc["text"])
    for doc in documents
]

# BM25 검색기를 만들기. 
bm25 = BM25Okapi(
    tokenized_corpus
)

'''

BM25 검색기 : 
"이 문서들에 어떤 단어들이 들어있는지 분석해서
나중에 질문이 들어오면
키워드가 잘 맞는 문서를 점수 순으로 찾아줄 준비를 해라"

'''

'\n\nBM25 검색기 : \n"이 문서들에 어떤 단어들이 들어있는지 분석해서\n나중에 질문이 들어오면\n키워드가 잘 맞는 문서를 점수 순으로 찾아줄 준비를 해라"\n\n'

각 doc의 text를 아래처럼 자름 :
```
[
    "도서관은",
    "오전",
    "9시부터",
    "운영합니다"
]
```

In [14]:
def metadata_matches(
    metadata,
    where
):

    if where is None:
        return True

    # 이번 실습에서는
    # {"section": "도서관"}
    # 같은 equality filter만 지원

    for key, value in where.items():

        if metadata.get(key) != value:
            return False

    return True

In [15]:

def bm25_search(
    question,
    top_k=10,
    where=None
):

    query_tokens = tokenize(question)

    # 모든 문서에 대해 질문에 대한 BM25 점수를 계산
    scores = bm25.get_scores(
        query_tokens
    )

    candidates = []

    for index, score in enumerate(scores):

        doc = documents[index]  # 원본 가져와서

        if not metadata_matches(
            doc["metadata"],
            where
        ):
            continue

        candidates.append(
            {
                **doc,
                "bm25_score": float(score)
            }
        )

    candidates.sort(
        key=lambda x: x["bm25_score"],
        reverse=True
    )

    return candidates[:top_k]

candidates 결과 예시 (앞에서부터 2개) :
```
[
    {
        "id": "chunk-002",
        "text": "도서 대출 기간은 최대 14일이다.",
        "metadata": {"section": "도서관"},
        "bm25_score": 4.2
    },
    {
        "id": "chunk-000",
        "text": "도서관은 오전 9시부터 오후 9시까지 운영한다.",
        "metadata": {"section": "도서관"},
        "bm25_score": 1.7
    }
]
```

**Chroma vs BM25**

Chroma Vector Search : 
질문과 문서의 "의미"가 비슷한가?


BM25 : 
질문과 문서의 "단어"가 잘 겹치는가?

=> 이 둘을 혼합한 Hybrid Search 가 좋음 (약점 보완)

## Level 5 : Hybrid Search
Vector Search + BM25 -> RRF (Reciprocal Rank Fusion)

RRF : Vector와 BM25의 단위와 범위가 다르기에, 
Vector에서 몇 위였는가? 
BM25에서 몇 위였는가? 
를 기준으로 합치는 거임.

```RRF 점수 = 1 / (K + rank)```

---
```
질문
  ↓
Vector Search
  ↓
벡터 검색 순위

질문
  ↓
BM25 Search
  ↓
키워드 검색 순위

두 순위를 RRF로 합침
  ↓
최종 Hybrid 순위
```

In [16]:
def hybrid_search(
    question,
    top_k=10,
    where=None
):

    vector_hits = vector_search(
        question,
        top_k=top_k,
        where=where
    )

    bm25_hits = bm25_search(
        question,
        top_k=top_k,
        where=where
    )

    # RRF (Reciprocal Rank Fusion) 점수 계산（두 검색 결과를 합칠 공간）
    fused = {}

    RRF_K = 60

    # ---------------------------------
    # Vector rank
    # ---------------------------------

    for rank, hit in enumerate(
        vector_hits,
        start=1
    ):

        doc_id = hit["id"]

        if doc_id not in fused:

            fused[doc_id] = {
                **hit,
                "rrf_score": 0
            }

        fused[doc_id]["rrf_score"] += (
            1 / (RRF_K + rank)
        )


    # ---------------------------------
    # BM25 rank
    # ---------------------------------

    for rank, hit in enumerate(
        bm25_hits,
        start=1
    ):

        doc_id = hit["id"]

        if doc_id not in fused:

            fused[doc_id] = {
                **hit,
                "rrf_score": 0
            }

        fused[doc_id]["rrf_score"] += (
            1 / (RRF_K + rank)
        )

    # 즉, rrf_score는 vector rank 점수와 bm25 rank 점수를 합한 것.

    results = list(
        fused.values()
    )

    results.sort(
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    return results[:top_k]

fused 예시
```
fused["chunk-002"] = {
    "id": "chunk-002",
    "text": "도서 대출 기간은 최대 14일이다.",
    "vector_distance": 0.08,
    "rrf_score": 0
}
```

In [17]:
scores = []

for chunk, embedding in zip(chunks, embeddings):
    score = cosine_similarity(question_embedding, embedding)
    scores.append((score, chunk))

scores.sort(reverse=True)

for score, chunk in scores:
    print(score, chunk)

NameError: name 'chunks' is not defined

## Level 6 : Reranker

보통 이런 프로세스 : 
```
RRF Fusion
      ↓
후보 10~30개
      ↓
Reranker
      ↓
최종 top 3~5개
```

Hybrid Search(RRF)로 뽑은 후보들을 CrossEncoder가 질문과 직접 비교해서 다시 순위를 매기는 단계

In [ ]:
_reranker = None

# CrossEncoder 불러오기
def get_reranker():

    global _reranker

    if _reranker is None:

        print(
            "\nReranker 모델 로딩..."
        )

        _reranker = CrossEncoder(
            RERANKER_MODEL
        )

    return _reranker


def rerank(
    question,
    candidates,
    top_k=3
):

    if not candidates:
        return []

    reranker = get_reranker()

    # 질문과 후보 문서 chunk를 쌍으로 만들어서
    pairs = [
        (
            question,
            candidate["text"]
        )
        for candidate in candidates
    ]

    # CrossEncoder에 넣어 rerank 점수를 계산
    scores = reranker.predict(
        pairs
    )

    results = []

    for candidate, score in zip(
        candidates,
        scores
    ):

        result = dict(candidate)

        result["rerank_score"] = float(
            np.asarray(score).squeeze()
        )

        results.append(result)

    # 결과를 rerank 점수 기준으로 내림차순 정렬
    results.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return results[:top_k]

### SentenceTransformer VS CrossEncoder


전체 문서에 처음부터 CrossEncoder를 쓰면 너무 느림.

이유는 계산 방식 차이예요.

SentenceTransformer + cosine은:

```
문서 100,000개

↓

문서 임베딩은 미리 계산해서 DB에 저장

질문 1개

↓

질문 임베딩 1번 계산

↓

벡터 DB가 빠르게 가까운 문서 검색
```

즉 문서 임베딩을 **재사용**할 수 있습니다.

반면 CrossEncoder는 질문과 문서를 매번 같이 넣어야 해요.

```
질문 + 문서1 → Transformer 실행

질문 + 문서2 → Transformer 실행

질문 + 문서3 → Transformer 실행

...

질문 + 문서100,000 → Transformer 실행
```

질문 하나 들어올 때마다 Transformer를 10만 번 수준으로 돌려야 하는 셈이라 너무 비쌉니다.

---

그래서 실제 RAG는 보통 **2단계 구조**를 씁니다.

```
전체 문서 100,000개

        ↓

Vector / BM25

빠르게 후보 20개 추림

        ↓

CrossEncoder

20개만 정밀 평가

        ↓

최종 3~5개
```

---

SentenceTransformer는:

```
문서 → embedding
```

을 **미리 만들어 Chroma에 저장 가능**합니다.

그런데 CrossEncoder는:

```
질문 + 문서 → score
```

이기 때문에 질문이 바뀌면 점수도 바뀝니다.

그래서 CrossEncoder 점수를 미리 DB에 저장해둘 수도 없습니다.

---

결론은:

```
CrossEncoder가 더 정밀

하지만 느림

Embedding + cosine

조금 덜 정밀할 수 있음

하지만 매우 빠름
```


## 6. Level 7 : Citation 만들기

이 답변이 어느 문서(chunk)를 근거로 만들었는지

In [ ]:
# 검색된 RAG 문서(hit)에 citation ID를 붙이고, LLM 프롬트에 넣을 context 문자열로 변환하는 역할

def attach_citations(
    hits
):

    cited = []

    # top k 검색 결과에 대해 citation ID를 붙임
    for index, hit in enumerate(
        hits,
        start=1
    ):

        item = dict(hit)

        item["citation_id"] = (
            f"C{index}"
        )

        cited.append(item)

    return cited


def build_context(
    cited_hits
):

    context_parts = []

    # 밑에 md에 예시처럼 문자열 형태로 저장
    for hit in cited_hits:

        citation = hit[
            "citation_id"
        ]

        section = hit[
            "metadata"
        ].get(
            "section",
            "Unknown"
        )

        source = hit[
            "metadata"
        ].get(
            "source",
            "Unknown"
        )

        context_parts.append(
            f"""
[{citation}]
source: {source}
section: {section}
text: {hit["text"]}
""".strip()
        )

    return "\n\n".join(
        context_parts
    )


```
[C1]
source: rag.md
section: RAG
text: RAG는 검색과 생성을 결합한다.

[C2]
source: search.md
section: BM25
text: BM25는 키워드 기반 검색 알고리즘이다.
```

## LLM

In [ ]:
_llm_client = None


def get_llm_client():

    global _llm_client

    if _llm_client is None:

        if not os.getenv(
            "OPENAI_API_KEY"
        ):
            raise RuntimeError(
                "OPENAI_API_KEY가 없습니다."
            )

        if not OPENAI_MODEL:
            raise RuntimeError(
                "OPENAI_MODEL을 .env에 설정하세요."
            )

        _llm_client = OpenAI()

    return _llm_client


def generate_answer(
    question,
    hits
):

    cited_hits = attach_citations(
        hits
    )

    context = build_context(
        cited_hits
    )

    client = get_llm_client()

    response = client.responses.create(

        model=OPENAI_MODEL,

        instructions="""
너는 RAG 기반 질의응답 시스템이다.

반드시 제공된 참고 문서만 사용해서 답변한다.

규칙:

1. 참고 문서에 없는 사실을 만들지 않는다.
2. 사실을 말할 때 반드시 [C1], [C2] 형태로 근거를 표시한다.
3. 관련 근거가 없다면 "제공된 문서에서 확인할 수 없습니다."라고 답한다.
4. citation 번호는 제공된 번호만 사용한다.
5. 짧고 명확하게 답한다.
""",

        input=f"""
[참고 문서]

{context}


[사용자 질문]

{question}
"""
    )

    return (
        response.output_text.strip(),
        cited_hits
    )



프롬프트 예시 : 
```
[참고 문서]

[C1]
source: school_guide.txt
section: 도서관
text: 학생은 한 번에 최대 5권까지 대출할 수 있다.

[C2]
source: school_guide.txt
section: 도서관
text: 도서 대출 기간은 최대 14일이다.

[C3]
source: school_guide.txt
section: 도서관
text: 학교 도서관은 오전 9시부터 오후 9시까지 운영한다.

[사용자 질문]

도서관에서 책을 몇 권까지 빌릴 수 있어?
```

답변 예시 : 
```
학생은 한 번에 최대 5권까지 대출할 수 있습니다. [C1]
```